# Document intelligence with Nemotron 3.5 Super VL

This notebook sends three synthetic business pages to `nvidia/nemotron-3.5-super-vl-120b-a12b` on the NVIDIA NIM API. The model reads the page. The notebook checks the JSON against `test_data/ground_truth.json`.

1. A tax invoice. Extract the vendor, dates, line items, and total.
2. A till receipt. Extract the basket and the amount paid.
3. A return form. Read the page, call `lookup_return_policy`, then decide if the return is inside the window.

The pages are fictional and generated by `scripts/make_test_data.py`. The return rule is not printed on the form. It lives in `test_data/policies.json`, so a correct decision has to use the tool.

Put your NVIDIA API key in a Colab secret named `NVIDIA_API_KEY`, or in the `NVIDIA_API_KEY` environment variable. The notebook never writes the key down.

Sampling follows the model card for vision: temperature `0.6`, top_p `0.95`, reasoning off (`chat_template_kwargs.enable_thinking` is false).


In [ ]:
%pip install -q openai pillow


In [ ]:
import base64
import getpass
import json
import os
import subprocess
from pathlib import Path

from IPython.display import Image, display
from openai import OpenAI


def load_api_key() -> str:
    try:
        from google.colab import userdata

        key = userdata.get("NVIDIA_API_KEY")
        if key and key.strip():
            return key.strip()
    except Exception:
        pass
    key = os.environ.get("NVIDIA_API_KEY", "").strip()
    if key:
        return key
    return getpass.getpass("NVIDIA API key: ").strip()


def find_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "test_data" / "ground_truth.json").exists():
            return candidate
    dest = Path("/content/nemotron-3.5-super-vl") if Path("/content").exists() else here / "nemotron-3.5-super-vl"
    if not (dest / "test_data" / "ground_truth.json").exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/cobusgreyling/nemotron-3.5-super-vl.git",
                str(dest),
            ]
        )
    return dest


ROOT = find_root()
TRUTH = json.loads((ROOT / "test_data" / "ground_truth.json").read_text())
POLICIES = json.loads((ROOT / "test_data" / "policies.json").read_text())
MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=load_api_key(),
)
print(f"repo: {ROOT}")
print(f"model: {MODEL}")


In [ ]:
def page_message(path: Path, prompt: str) -> dict:
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/png;base64,{encoded}"},
            },
        ],
    }


def complete(messages: list, tools: list | None = None):
    kwargs = {
        "model": MODEL,
        "messages": messages,
        "temperature": 0.6,
        "top_p": 0.95,
        "max_tokens": 1500,
        "seed": 42,
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    }
    if tools is not None:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    return client.chat.completions.create(**kwargs)


def parse_json(content: str) -> dict:
    raw = (content or "").strip()
    if raw.startswith("```"):
        raw = raw.split("```", 2)[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start < 0 or end < start:
        raise ValueError(raw[:500] or "The model returned an empty message.")
    return json.loads(raw[start : end + 1])


def norm(value):
    if isinstance(value, bool) or value is None:
        return value
    if isinstance(value, (int, float)):
        return round(float(value), 2)
    if isinstance(value, str):
        return " ".join(value.split()).casefold()
    if isinstance(value, list):
        return [norm(item) for item in value]
    if isinstance(value, dict):
        return {key: norm(item) for key, item in value.items()}
    return value


def score(expected: dict, actual: dict, skip: set[str] | None = None) -> None:
    skipped = skip or set()
    rows = []
    for key, exp in expected.items():
        if key in skipped:
            continue
        got = actual.get(key, "<missing>")
        rows.append((key, norm(got) == norm(exp), exp, got))
    matched = sum(1 for _, ok, _, _ in rows if ok)
    print(f"{matched}/{len(rows)} fields match")
    for key, ok, exp, got in rows:
        mark = "ok  " if ok else "DIFF"
        print(f"{mark} {key}")
        if not ok:
            print(f"     expected: {exp}")
            print(f"     got:      {got}")


def show_page(name: str) -> None:
    path = ROOT / "test_data" / name
    display(Image(filename=str(path), width=420))


## 1. Tax invoice

The page is a one-page supplier invoice. The check covers the identity fields, both line items, VAT, and the total.


In [ ]:
invoice_prompt = '''
Read this tax invoice. Return only a JSON object with these keys:
document_type, vendor, vendor_vat, invoice_number, invoice_date, due_date,
purchase_order, currency, bill_to, line_items, subtotal, vat_rate, vat_amount, total.

bill_to is the customer name as one string, not an object.

Use document_type "tax_invoice".
Dates are YYYY-MM-DD.
Money fields are JSON numbers, not strings.
vat_rate is a decimal, so 15% is 0.15.
line_items is a list of objects with sku, description, quantity, unit_price, amount.
Copy only values printed on the page.
'''.strip()

show_page("invoice.png")
invoice_response = complete([page_message(ROOT / "test_data" / "invoice.png", invoice_prompt)])
invoice_text = invoice_response.choices[0].message.content or ""
invoice = parse_json(invoice_text)
print(json.dumps(invoice, indent=2))
score(TRUTH["invoice"], invoice, skip={"file"})
print("tokens", invoice_response.usage.total_tokens)


## 2. Till receipt

The merchant, the three lines, the total, and the VAT already included in that total are all printed on the slip.


In [ ]:
receipt_prompt = '''
Read this till receipt. Return only a JSON object with these keys:
merchant, vat_number, receipt_number, sold_at, currency, line_items, total,
vat_included, tender.

sold_at is an ISO local datetime, YYYY-MM-DDTHH:MM:SS, with no timezone suffix.
line_items is a list of objects with description and amount.
Money fields are JSON numbers.
tender is lower case, such as "card".
currency is the three-letter code printed with the total.
Copy only values printed on the slip.
'''.strip()

show_page("receipt.png")
receipt_response = complete([page_message(ROOT / "test_data" / "receipt.png", receipt_prompt)])
receipt = parse_json(receipt_response.choices[0].message.content or "")
print(json.dumps(receipt, indent=2))
score(TRUTH["receipt"], receipt, skip={"file"})
print("tokens", receipt_response.usage.total_tokens)


## 3. Return form, then the policy tool

The form names the customer, the order, the product, and the two dates. It does not state the return window. `lookup_return_policy` does.

Days since purchase means the request date minus the purchase date. For this form that is 24 days. Apparel has a 30-day window, and a defective item is covered inside that window, so the return is eligible.


In [ ]:
RETURN_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_return_policy",
            "description": "Look up the return rule for one product category.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "enum": ["apparel", "electronics", "office"],
                        "description": "Category printed on the form, in lower case.",
                    }
                },
                "required": ["category"],
            },
        },
    }
]


def lookup_return_policy(category: str) -> dict:
    key = category.strip().lower()
    rule = POLICIES.get(key)
    if rule is None:
        return {"error": f"No policy for {category!r}", "known": sorted(POLICIES)}
    return {"category": key, **rule}


return_prompt = '''
Read this return request. The return window is not on the page.
Call lookup_return_policy with the category printed on the form before you decide.
Days since purchase is the request date minus the purchase date.
When you have the tool result, return only a JSON object with these keys:
order_number, category, days_since_purchase, policy_window_days, eligible, reason.

category is lower case.
days_since_purchase and policy_window_days are integers.
eligible is a JSON boolean.
reason is one sentence.
'''.strip()

show_page("return_request.png")
messages = [page_message(ROOT / "test_data" / "return_request.png", return_prompt)]
tool_trace = []
decision = None
return_tokens = 0

for _ in range(4):
    response = complete(messages, tools=RETURN_TOOLS)
    return_tokens += response.usage.total_tokens
    message = response.choices[0].message
    calls = message.tool_calls or []
    if not calls:
        decision = parse_json(message.content or "")
        break
    messages.append(message)
    for call in calls:
        arguments = json.loads(call.function.arguments or "{}")
        if call.function.name != "lookup_return_policy":
            result = {"error": f"Unknown tool {call.function.name}"}
        else:
            result = lookup_return_policy(arguments.get("category", ""))
        tool_trace.append({"arguments": arguments, "result": result})
        messages.append(
            {
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result),
            }
        )

print("tool calls")
print(json.dumps(tool_trace, indent=2))
print("decision")
print(json.dumps(decision, indent=2))

expected = {
    "order_number": TRUTH["return_request"]["order_number"],
    **TRUTH["return_request"]["expected_decision"],
}
score(expected, decision or {}, skip={"reason"})
if not tool_trace:
    print("DIFF no policy tool call")
print("tokens", return_tokens)


## What a match means

`ok` means the printed value and the model value normalize to the same number or string. A `DIFF` line prints both. The return check ignores the wording of `reason` and requires the tool call, the order number, the 24-day gap, the 30-day window, and `eligible: true`.

Regenerate the pages with `python scripts/make_test_data.py` if you change a fixture. Update `test_data/ground_truth.json` in the same change. The generator needs the Arial and Courier New fonts shipped with macOS.
